# mammo-cad — Explainability : Grad-CAM++ + Deep SHAP

| Method | Ce qu'il montre |
|---|---|
| **Grad-CAM++** | WHERE le modèle regarde — carte d'activation spatiale, meilleure localisation que Grad-CAM sur les petites structures |
| **Deep SHAP** | WHY chaque pixel contribue — attribution signée : **rouge** = pousse vers MALIGNANT, **bleu** = pousse vers BENIGN |

**Target layer Grad-CAM++ :** `net.features[7]` — dernier bloc MBConv avant GlobalAvgPool, résolution spatiale 7×7.

In [1]:
import sys
import numpy as np
import cv2
import torch
import matplotlib
matplotlib.rcParams['figure.dpi'] = 130
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.inference.classify import MammoClassifier
from src.inference.explainability import (
    ExplainabilityEngine, plot_xai_panel, explain_pipeline_results
)

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {device}')

# SHAP check
try:
    import shap
    print(f'SHAP    : {shap.__version__} ✅')
    USE_SHAP = True
except ImportError:
    print('SHAP    : NOT INSTALLED — run: pip install shap')
    USE_SHAP = False

PyTorch : 2.11.0+cpu
CUDA    : False
Device  : cpu
SHAP    : 0.49.1 ✅


In [2]:
# ── Configuration ─────────────────────────────────────────────────────
CLS_CKPT   = PROJECT_ROOT / 'checkpoints' / 'best_0.7717' / 'efficientnet_stage2.pth'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'xai'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Images à expliquer — crops 224×224 déjà extraits par le pipeline
# Ou n'importe quelle image 224×224 grayscale
CROPS_DIR  = PROJECT_ROOT / 'outputs' / 'predictions'

# Charger le modèle
clf = MammoClassifier(CLS_CKPT, device=str(device), tta=1)  # TTA=1 pour XAI
print(f'\nModèle chargé — Val AUC : {clf.val_auc:.4f}')

  Loaded  : efficientnet_stage2.pth
  Stage   : 2
  Val AUC : 0.8056
  Device  : cpu
  TTA     : 1 views
  Threshold (val-optimised): 0.50

Modèle chargé — Val AUC : 0.8056


## 1 · Charger les crops à expliquer

In [3]:
# Chercher tous les crops générés par le pipeline
crop_paths = sorted(CROPS_DIR.rglob('*_crop.png'))
print(f'Crops trouvés : {len(crop_paths)}')
for p in crop_paths:
    print(f'  {p.relative_to(PROJECT_ROOT)}')

# Si pas de crops du pipeline, utiliser des images du test set
if not crop_paths:
    print('\nAucun crop pipeline trouvé — utilisation du test set...')
    TEST_DIR   = PROJECT_ROOT / 'data' / 'patches' / 'classification' / 'test'
    mal_crops  = sorted((TEST_DIR / 'malignant').glob('*.png'))[:3]
    ben_crops  = sorted((TEST_DIR / 'benign').glob('*.png'))[:3]
    crop_paths = mal_crops + ben_crops
    print(f'  {len(crop_paths)} images du test set chargées')

Crops trouvés : 25
  outputs\predictions\crops\20587294_e634830794f5c1bd_MG_R_CC_ANON_R01_crop.png
  outputs\predictions\crops\20587294_e634830794f5c1bd_MG_R_CC_ANON_R02_crop.png
  outputs\predictions\crops\20587294_e634830794f5c1bd_MG_R_CC_ANON_R03_crop.png
  outputs\predictions\crops\20587612_f4b2d377f43ba0bd_MG_R_CC_ANON_R01_crop.png
  outputs\predictions\crops\20587612_f4b2d377f43ba0bd_MG_R_CC_ANON_R02_crop.png
  outputs\predictions\crops\20587612_f4b2d377f43ba0bd_MG_R_CC_ANON_R03_crop.png
  outputs\predictions\crops\20587612_f4b2d377f43ba0bd_MG_R_CC_ANON_R04_crop.png
  outputs\predictions\crops\20587612_f4b2d377f43ba0bd_MG_R_CC_ANON_R05_crop.png
  outputs\predictions\crops\20587612_f4b2d377f43ba0bd_MG_R_CC_ANON_R06_crop.png
  outputs\predictions\crops\20587612_f4b2d377f43ba0bd_MG_R_CC_ANON_R07_crop.png
  outputs\predictions\crops\20587612_f4b2d377f43ba0bd_MG_R_CC_ANON_R08_crop.png
  outputs\predictions\crops\20587612_f4b2d377f43ba0bd_MG_R_CC_ANON_R09_crop.png
  outputs\predictions

In [4]:
# Charger les crops et obtenir les prédictions
crops   = []
metas   = []

for i, p in enumerate(crop_paths[:6]):  # max 6 pour la lisibilité
    img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue
    if img.shape != (224, 224):
        img = cv2.resize(img, (224, 224))

    prob, label_int, per_tta = clf.predict_array(img, return_all_probs=True)
    label = 'MALIGNANT' if label_int == 1 else 'BENIGN'
    uncert = float(np.std(per_tta))

    # Essayer de lire le label depuis le chemin
    if 'malignant' in str(p).lower():
        gt = 'MALIGNANT'
    elif 'benign' in str(p).lower():
        gt = 'BENIGN'
    else:
        gt = '?'

    correct = '✅' if label == gt or gt == '?' else '❌'
    print(f'  [{i+1}] {p.name[:40]:40s}  '
          f'GT={gt:9s}  Pred={label:9s}  P={prob:.3f}  {correct}')

    crops.append(img)
    metas.append(dict(
        region_id   = i + 1,
        label       = label,
        prob        = prob,
        uncertainty = uncert,
        gt          = gt,
        path        = str(p),
    ))

print(f'\n{len(crops)} images prêtes pour XAI')

  [1] 20587294_e634830794f5c1bd_MG_R_CC_ANON_R  GT=?          Pred=BENIGN     P=0.322  ✅
  [2] 20587294_e634830794f5c1bd_MG_R_CC_ANON_R  GT=?          Pred=BENIGN     P=0.298  ✅
  [3] 20587294_e634830794f5c1bd_MG_R_CC_ANON_R  GT=?          Pred=BENIGN     P=0.383  ✅
  [4] 20587612_f4b2d377f43ba0bd_MG_R_CC_ANON_R  GT=?          Pred=MALIGNANT  P=0.518  ✅
  [5] 20587612_f4b2d377f43ba0bd_MG_R_CC_ANON_R  GT=?          Pred=BENIGN     P=0.157  ✅
  [6] 20587612_f4b2d377f43ba0bd_MG_R_CC_ANON_R  GT=?          Pred=BENIGN     P=0.493  ✅

6 images prêtes pour XAI


## 2 · Initialisation du moteur XAI

In [5]:
engine = ExplainabilityEngine(
    model    = clf.model,
    device   = device,
    use_shap = USE_SHAP,
)
print(f'Engine créé')
print(f'  Grad-CAM++ : ✅ (target: net.features[7])')
print(f'  Deep SHAP  : {"✅" if USE_SHAP else "❌ (pip install shap)"}')

  [XAI] Integrated Gradients ready (captum)
Engine créé
  Grad-CAM++ : ✅ (target: net.features[7])
  Deep SHAP  : ✅


## 3 · Calcul des explications

In [6]:
%%time
xai_results = []
for i, (crop, meta) in enumerate(zip(crops, metas)):
    print(f'[{i+1}/{len(crops)}] R{meta["region_id"]:02d} '
          f'{meta["label"]} P={meta["prob"]:.3f}...')
    result = engine.explain(crop)
    xai_results.append(result)

engine.remove_hooks()
print(f'\n✅ {len(xai_results)} explications calculées')

[1/6] R01 BENIGN P=0.322...
[2/6] R02 BENIGN P=0.298...
[3/6] R03 BENIGN P=0.383...
[4/6] R04 MALIGNANT P=0.518...
[5/6] R05 BENIGN P=0.157...
[6/6] R06 BENIGN P=0.493...

✅ 6 explications calculées
CPU times: total: 3min 39s
Wall time: 34.3 s


## 4 · Figure principale — panel professionnel

In [7]:
out_path = OUTPUT_DIR / 'xai_panel_full.png'
plot_xai_panel(
    regions_data = xai_results,
    region_metas = metas,
    output_path  = out_path,
    title        = 'Explainability — EfficientNet-B3 | Grad-CAM++ + Deep SHAP',
    dpi          = 160,
)
print(f'Sauvegardé : {out_path}')

  [XAI] Panel saved → C:\project\mammo-cad\outputs\xai\xai_panel_full.png
Sauvegardé : C:\project\mammo-cad\outputs\xai\xai_panel_full.png


## 5 · Comparaison Correct vs Incorrect

In [8]:
# Séparer correct / incorrect
correct_idx   = [i for i, m in enumerate(metas)
                 if m['gt'] == '?' or m['label'] == m['gt']]
incorrect_idx = [i for i, m in enumerate(metas)
                 if m['gt'] != '?' and m['label'] != m['gt']]

print(f'Correct   : {len(correct_idx)}')
print(f'Incorrect : {len(incorrect_idx)}')

if correct_idx and incorrect_idx:
    # Comparer 2 corrects et 2 incorrects
    sel_idx = correct_idx[:2] + incorrect_idx[:2]
    sel_res = [xai_results[i] for i in sel_idx]
    sel_met = [metas[i]       for i in sel_idx]
    # Add correct/incorrect marker to title
    for i, (m, orig_i) in enumerate(zip(sel_met, sel_idx)):
        tag = '✅ CORRECT' if orig_i in correct_idx else '❌ WRONG'
        sel_met[i] = {**m, 'label': f"{m['label']} [{tag}]"}

    plot_xai_panel(
        sel_res, sel_met,
        output_path = OUTPUT_DIR / 'xai_correct_vs_wrong.png',
        title       = 'XAI — Correct vs Wrong Predictions',
    )
else:
    print('Pas assez de cas pour la comparaison correct/incorrect.')

Correct   : 6
Incorrect : 0
Pas assez de cas pour la comparaison correct/incorrect.


## 6 · Analyse des cartes d'activation — statistiques

In [9]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), facecolor='#0d1117')

mal_gcpp = [r['gradcampp'] for r, m in zip(xai_results, metas)
            if m['label'] == 'MALIGNANT']
ben_gcpp = [r['gradcampp'] for r, m in zip(xai_results, metas)
            if m['label'] == 'BENIGN']

# ── [0,0] Distribution des scores d'activation ───────────────────────
ax = axes[0][0]
ax.set_facecolor('#161b22')
for maps, color, label in [
    (mal_gcpp, '#FF4444', 'MALIGNANT'),
    (ben_gcpp, '#4DA1FF', 'BENIGN'),
]:
    if maps:
        vals = np.concatenate([m.flatten() for m in maps])
        ax.hist(vals, bins=50, color=color, alpha=0.7,
                label=f'{label} (n={len(maps)})', density=True)
ax.axvline(0.5, color='white', ls='--', lw=1, label='thr=0.5')
ax.set_xlabel('Activation Grad-CAM++', color='white')
ax.set_ylabel('Densité', color='white')
ax.set_title('Distribution activation\nMAL vs BEN', color='#aaa', fontsize=9)
ax.legend(fontsize=7); ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_color('#444')

# ── [0,1] Activation moyenne par classe ──────────────────────────────
ax = axes[0][1]
ax.set_facecolor('#161b22')
if mal_gcpp:
    avg_mal = np.mean(np.stack(mal_gcpp), axis=0)
    ax.imshow(avg_mal, cmap='inferno', vmin=0, vmax=1)
    ax.set_title('Activation moyenne\nMALIGNANT', color='#FF4444', fontsize=9)
ax.axis('off')

ax = axes[0][2]
ax.set_facecolor('#161b22')
if ben_gcpp:
    avg_ben = np.mean(np.stack(ben_gcpp), axis=0)
    ax.imshow(avg_ben, cmap='inferno', vmin=0, vmax=1)
    ax.set_title('Activation moyenne\nBENIGN', color='#4DA1FF', fontsize=9)
ax.axis('off')

# ── [1,0] Surface activée (%) ─────────────────────────────────────────
ax = axes[1][0]
ax.set_facecolor('#161b22')
for maps, color, label in [
    (mal_gcpp, '#FF4444', 'MAL'),
    (ben_gcpp, '#4DA1FF', 'BEN'),
]:
    areas = [(m >= 0.5).mean() * 100 for m in maps]
    if areas:
        ax.bar([label], [np.mean(areas)],
               yerr=[np.std(areas)],
               color=color, alpha=0.8, capsize=5, width=0.4)
        ax.text(label, np.mean(areas) + np.std(areas) + 0.5,
                f'{np.mean(areas):.1f}%',
                ha='center', color='white', fontsize=9)
ax.set_ylabel('Surface activée (thr≥0.5) %', color='white')
ax.set_title('Surface activation\nMAL vs BEN', color='#aaa', fontsize=9)
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_color('#444')

# ── [1,1] Max activation ──────────────────────────────────────────────
ax = axes[1][1]
ax.set_facecolor('#161b22')
for maps, color, label in [
    (mal_gcpp, '#FF4444', 'MAL'),
    (ben_gcpp, '#4DA1FF', 'BEN'),
]:
    maxes = [m.max() for m in maps]
    if maxes:
        ax.bar([label], [np.mean(maxes)],
               yerr=[np.std(maxes)],
               color=color, alpha=0.8, capsize=5, width=0.4)
ax.set_ylabel('Activation max', color='white')
ax.set_title('Peak activation\nMAL vs BEN', color='#aaa', fontsize=9)
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_color('#444')

# ── [1,2] SHAP positif vs négatif ─────────────────────────────────────
ax = axes[1][2]
ax.set_facecolor('#161b22')
shap_available = any(r['shap'] is not None for r in xai_results)
if shap_available:
    for maps_idx, color, label in [
        ([i for i, m in enumerate(metas) if m['label']=='MALIGNANT'],
         '#FF4444', 'MAL'),
        ([i for i, m in enumerate(metas) if m['label']=='BENIGN'],
         '#4DA1FF', 'BEN'),
    ]:
        pos_ratios = []
        for i in maps_idx:
            s = xai_results[i]['shap']
            if s is not None:
                pos_ratios.append((s > 0).mean() * 100)
        if pos_ratios:
            ax.bar([label], [np.mean(pos_ratios)],
                   color=color, alpha=0.8, width=0.4)
            ax.text(label, np.mean(pos_ratios) + 1,
                    f'{np.mean(pos_ratios):.1f}%',
                    ha='center', color='white', fontsize=9)
    ax.set_ylabel('% pixels SHAP positif', color='white')
    ax.set_title('SHAP positif (→malignant)\n%', color='#aaa', fontsize=9)
else:
    ax.text(0.5, 0.5, 'SHAP N/A\npip install shap',
            ha='center', va='center', color='#555',
            transform=ax.transAxes, fontsize=10)
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_color('#444')

plt.suptitle('Analyse statistique des cartes XAI',
             color='white', fontsize=12)
plt.tight_layout()
out = OUTPUT_DIR / 'xai_statistics.png'
plt.savefig(str(out), dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f'Sauvegardé : {out}')

Sauvegardé : C:\project\mammo-cad\outputs\xai\xai_statistics.png


## 7 · Récapitulatif des fichiers générés

In [10]:
print(f'Output : {OUTPUT_DIR}\n')
for f in sorted(OUTPUT_DIR.rglob('*')):
    if f.is_file():
        print(f'  {f.name}  ({f.stat().st_size/1e3:.0f} KB)')

Output : C:\project\mammo-cad\outputs\xai

  xai_panel_full.png  (5323 KB)
  xai_panel_smoothed.png  (5636 KB)
  xai_statistics.png  (213 KB)


In [11]:
from scipy.ndimage import gaussian_filter
from src.inference.explainability import _make_shap_overlay

for r in xai_results:
    if r['shap'] is not None:
        r['shap'] = gaussian_filter(r['shap'], sigma=2.0)
        r['overlay_shap'] = _make_shap_overlay(r['crop'], r['shap'])

plot_xai_panel(
    xai_results, metas,
    output_path = OUTPUT_DIR / 'xai_panel_smoothed.png',
    title       = 'Explainability — Grad-CAM++ + Integrated Gradients (smoothed)',
    dpi         = 160,
)

  [XAI] Panel saved → C:\project\mammo-cad\outputs\xai\xai_panel_smoothed.png
